# Demo nhận dạng chữ viết tay theo dòng (VietHandOCR)

Cách dùng:

1. Đặt ảnh vào thư mục `images/` cạnh notebook này. **Mỗi ảnh chỉ chứa một dòng chữ**, cắt sát bằng app điện thoại. Ảnh cả trang sẽ cho kết quả sai, vì model chỉ đọc từng dòng (chưa có bước cắt dòng).
2. Chọn kernel Python đã cài `requirements-notebook.txt` (xem `README.md` của gói model). Trên máy nhóm, vẫn chọn kernel `.venv` như trước.
3. Chạy **Ô 1** một lần để nạp thư viện.
4. Sửa danh sách `IMAGE_PATHS` trong **Ô 2** rồi chạy Ô 2. Muốn thử ảnh khác thì sửa danh sách và chạy lại Ô 2; model đã nạp được giữ lại, không nạp lại.
5. Bật/tắt cắt lề V2c bằng biến `USE_V2C` trong Ô 2: `True` cắt lề trước khi nhận diện, giống hệt lúc train; `False` đưa ảnh gốc vào. Muốn xem cả hai cạnh nhau thì đặt `COMPARE_BOTH = True` (đang bật).

V2c được làm cho ảnh scan kiểu UIT (nền sáng, đều). Với ảnh chụp có bóng hoặc viền tối, V2c có thể coi vùng tối là mực và cắt không sát; khi khung cắt chạm mép ảnh, notebook ghi cảnh báo. `SHOW_V2C_STEPS = True` (mặc định) vẽ từng bước V2c cắt ảnh ngay dưới kết quả nhận diện: khung cắt, mặt nạ mực Otsu, histogram độ sáng với ngưỡng, ảnh sau khi cắt và ảnh đúng như model nhìn thấy. Sau khi sửa notebook, chạy lại Ô 1 một lần.

Ảnh hỗ trợ: `.jpg`, `.jpeg`, `.png`. Ảnh iPhone dạng `.heic` cần đổi sang JPG trước.

Model mặc định là `ML_s44`: VietOCR ảnh cao 64 px, học thêm với lề ngẫu nhiên, 8.000 bước, seed 44, chọn theo tập validation. Chạy trên CPU với 2 luồng: một dòng dài mất khoảng 1–3 giây. Notebook chạy được ở hai nơi: trong gói model, và trong thư mục `demo` trên máy nhóm. Trên máy nhóm có thêm model cũ để so (`P1H_s44`, `A`...) và notebook từ chối ảnh trong `data/raw/` của repo để không vô tình xem tập test.

In [ ]:
# Ô 1 — Chuẩn bị (chạy một lần)
from pathlib import Path
import json
import sys
import textwrap
import time

PACKAGE_NAME = "VietHandOCR_ML_s44_v1"
DEMO_DIR = Path.cwd()
PACKAGE_DIR = next((folder for folder in (DEMO_DIR, DEMO_DIR.parent, DEMO_DIR.parent / "model_ban_giao" / PACKAGE_NAME)
                    if (folder / "viethandocr_lite.py").is_file()), None)
if PACKAGE_DIR is None:
    raise FileNotFoundError("Không thấy viethandocr_lite.py. Hãy mở notebook ngay trong thư mục gói model.")
if str(PACKAGE_DIR) not in sys.path:
    sys.path.insert(0, str(PACKAGE_DIR))

import matplotlib.pyplot as plt
import torch

import numpy as np
from matplotlib.patches import Rectangle

from viethandocr_lite import (LineRecognizer, cer as line_cer, ink_bbox, ink_bbox_v2, ink_components, ink_mask,
                              load_image as read_image, normalize, prepare_image, sha256_file, v2c_touches_border)

# Máy nhóm có repo: thêm model cũ để so và chặn ảnh bộ UIT. Máy khác bỏ qua phần này.
REPO = (DEMO_DIR.parent / "hai-code" / "VietHandOCR").resolve()
REGISTRY = {"ML_s44": None}
RAW_DATA = None
if (REPO / "configs/research/p3_models.json").is_file():
    REGISTRY.update(json.loads((REPO / "configs/research/p3_models.json").read_text(encoding="utf-8")))
    RAW_DATA = (REPO / "data/raw").resolve()
_runners = {}


def repo_model(spec, device):
    """Model cũ trong repo: kiểm SHA checkpoint/config rồi mới nạp."""
    import yaml
    for key in ("checkpoint", "config"):
        if sha256_file(REPO / spec[key]) != spec[key + "_sha256"]:
            raise ValueError("SHA sai: " + spec[key])
    config = yaml.safe_load((REPO / spec["config"]).read_text(encoding="utf-8"))
    settings = {key: config["dataset"][key] for key in ("image_height", "image_min_width", "image_max_width")}
    settings.update(vocab=config["vocab"], cnn=config["cnn"], transformer=config["transformer"])
    return LineRecognizer.from_settings(settings, REPO / spec["checkpoint"], device)


def get_runner(name, device):
    """Nạp model một lần, rồi giữ lại."""
    if name not in REGISTRY:
        raise ValueError(f"Không có model {name!r}. Chọn một trong: {', '.join(REGISTRY)}")
    if (name, device) not in _runners:
        tick = time.perf_counter()
        _runners[(name, device)] = (LineRecognizer(PACKAGE_DIR / "model", device) if REGISTRY[name] is None
                                    else repo_model(REGISTRY[name], device))
        print(f"Đã nạp {name} trên {device} trong {time.perf_counter() - tick:.1f} giây")
    return _runners[(name, device)]


def load_image(path_text):
    path = Path(path_text)
    path = (path if path.is_absolute() else DEMO_DIR / path).resolve()
    if RAW_DATA and path.is_relative_to(RAW_DATA):
        raise ValueError(f"Không dùng ảnh trong data/raw/ (bộ UIT): {path_text}")
    if not path.is_file():
        raise FileNotFoundError(f"Không thấy ảnh: {path}")
    return path, read_image(path)


def show_result(title, image, lines, inputs=()):
    """Vẽ ảnh gốc với kết quả ở trên; thêm ảnh đầu vào model nếu có."""
    wrapped = [textwrap.fill(line, 100) for line in lines]
    text_rows = sum(w.count("\n") + 1 for w in wrapped)
    panels = [(None, image), *inputs]
    heights = [min(8.0, max(1.0, 13 * im.height / im.width)) for _, im in panels]
    heights[0] += 0.3 * text_rows
    fig, axes = plt.subplots(len(panels), 1, figsize=(13, sum(heights) + 0.6),
                             gridspec_kw=dict(height_ratios=heights), squeeze=False)
    for ax, (name, im) in zip(axes[:, 0], panels):
        ax.imshow(im)
        ax.axis("off")
        if name:
            ax.set_title(name, loc="left", fontsize=10, color="gray")
    axes[0, 0].set_title(title + "\n" + "\n".join(wrapped), loc="left", fontsize=12)
    fig.tight_layout()
    plt.show()



def v2c_steps(image):
    """Tính lại từng bước V2c (cùng hàm với prepare_image) để vẽ ra."""
    gray = np.asarray(image.convert("L"))
    mask = ink_mask(image)                       # điểm ảnh tối hơn hoặc bằng ngưỡng Otsu = "mực"
    core, box = ink_bbox(image), ink_bbox_v2(image)
    inside = np.zeros_like(mask)
    if box:
        left, top, right, bottom = box
        inside[top:bottom, left:right] = True
    labels, _ = ink_components(image)            # các mảng mực liền nhau
    edge_labels = np.unique(np.concatenate([labels[0], labels[-1], labels[:, 0], labels[:, -1]]))
    at_edge = np.isin(labels, edge_labels[edge_labels > 0])
    overlay = np.full((*mask.shape, 3), 255, np.uint8)
    overlay[mask] = (0, 0, 0)                     # mực bình thường: đen
    overlay[mask & ~inside] = (30, 110, 220)      # mực bị bỏ vì nằm ngoài khung V2c: xanh
    overlay[at_edge] = (220, 30, 30)              # mảng mực chạm mép ảnh: đỏ (thường là bóng, viền tối)
    ink = max(1, mask.sum())
    return dict(gray=gray, threshold=int(gray[mask].max()) if mask.any() else None, core=core, box=box,
                overlay=overlay, edge_share=at_edge.sum() / ink, outside_share=(mask & ~inside & ~at_edge).sum() / ink)


def draw_box(ax, box, color, style="-"):
    left, top, right, bottom = box
    ax.add_patch(Rectangle((left - 0.5, top - 0.5), right - left, bottom - top,
                           fill=False, edgecolor=color, linewidth=2, linestyle=style))


def show_v2c_result(title, image, lines, prepared, audit, inputs):
    """Một hình: kết quả nhận diện ở trên cùng, rồi từng bước V2c cắt ảnh, cuối cùng là ảnh model nhìn thấy."""
    steps = v2c_steps(image)
    wrapped = [textwrap.fill(line, 100) for line in lines]
    text_rows = sum(w.count("\n") + 1 for w in wrapped) + 2

    def ratio(im):
        return min(6.0, max(0.8, 13 * im.height / im.width))

    heights = [ratio(image) + 0.3 * text_rows, ratio(image), 2.2, ratio(prepared), *(ratio(im) + 0.3 for _, im in inputs)]
    fig, axes = plt.subplots(len(heights), 1, figsize=(13, sum(heights) + 1.5),
                             gridspec_kw=dict(height_ratios=heights), squeeze=False)
    axes = axes[:, 0]

    ax = axes[0]
    ax.imshow(image)
    ax.axis("off")
    if steps["core"]:
        draw_box(ax, steps["core"], "orange", "--")
    if steps["box"]:
        draw_box(ax, steps["box"], "red")
    ax.set_title(title + "\n" + "\n".join(wrapped) + "\n\n① Ảnh gốc. Khung cam (nét đứt) = lõi có nhiều mực; "
                 "khung đỏ = khung V2c (lõi + dấu, chân chữ dính gần)", loc="left", fontsize=11)

    ax = axes[1]
    ax.imshow(steps["overlay"])
    ax.axis("off")
    if steps["box"]:
        draw_box(ax, steps["box"], "red")
    ax.set_title(f"② Mặt nạ mực Otsu (khung đỏ = khung V2c). Đen = mực.\n"
                 f"Đỏ = mảng chạm mép ảnh, thường là bóng/viền ({100 * steps['edge_share']:.1f}% số điểm mực); "
                 f"xanh = mực bị bỏ ngoài khung "
                 f"({100 * steps['outside_share']:.1f}%)", loc="left", fontsize=10)

    ax = axes[2]
    ax.hist(steps["gray"].ravel(), bins=256, range=(0, 256), color="dimgray")
    ax.set_yscale("log")
    if steps["threshold"] is not None:
        ax.axvline(steps["threshold"] + 0.5, color="red")
        ax.axvspan(-0.5, steps["threshold"] + 0.5, color="red", alpha=0.1)
    ax.set_xlim(-0.5, 255.5)
    ax.set_xlabel("độ sáng (0 = đen, 255 = trắng)")
    ax.set_ylabel("số điểm ảnh (log)")
    ax.set_title(f"③ Histogram độ sáng. Ngưỡng Otsu = {steps['threshold']}: mọi điểm bên trái vạch đỏ bị coi là mực",
                 loc="left", fontsize=10)

    ax = axes[3]
    ax.imshow(prepared)
    ax.axis("off")
    if audit and audit["bbox_found"]:
        width, height = audit["content_size"]
        draw_box(ax, (audit["lr_px"], audit["tb_px"], audit["lr_px"] + width, audit["tb_px"] + height), "dodgerblue")
        ax.set_title(f"④ Sau V2c: cắt theo khung đỏ, kéo rộng {width} px, thêm lề {audit['lr_px']} px màu nền "
                     f"(khung xanh = phần chữ). Kích thước {prepared.width}×{prepared.height}", loc="left", fontsize=10)
    else:
        ax.set_title("④ V2c không thấy mực nên giữ nguyên ảnh", loc="left", fontsize=10)

    for ax, (name, im) in zip(axes[4:], inputs):
        ax.imshow(im)
        ax.axis("off")
        ax.set_title(f"⑤ {name}: cao {im.height} px, rộng {im.width} px", loc="left", fontsize=10)
    fig.tight_layout()
    plt.show()


print("Gói model:", PACKAGE_DIR)
print("Model có sẵn:", ", ".join(REGISTRY))
print("Ảnh trong images/:", ", ".join(sorted(p.name for p in (DEMO_DIR / "images").iterdir() if p.is_file())) or "(trống)")


In [ ]:
# Ô 2 — Sửa danh sách ảnh rồi chạy ô này. Đường dẫn tương đối tính từ thư mục demo.
IMAGE_PATHS = [
    "images/Ngon_Sang_tao_tre_quoc_gia.png",
]
MODELS = ["ML_s44"]       # trên máy nhóm có thể thêm model cũ để so, ví dụ ["ML_s44", "P1H_s44", "A"]
USE_V2C = True            # True = cắt lề V2c trước khi nhận diện (như lúc train); False = ảnh gốc
COMPARE_BOTH = True      # True = chạy cả ảnh gốc lẫn V2c để so, bỏ qua USE_V2C
LABELS = {}               # tùy chọn, để tính CER: {"images/co_be_ban_diem.png": "cô bé bán diêm"}
DEVICE = "cpu"            # "cuda" nếu máy có GPU NVIDIA
THREADS = 2               # ít luồng thì máy đỡ nóng nhưng chậm hơn
SHOW_MODEL_INPUT = True  # True để xem thêm ảnh đúng như model nhìn thấy
SHOW_V2C_STEPS = True     # True = vẽ từng bước V2c cắt ảnh ngay dưới kết quả nhận diện

PRE = ["raw", "v2c"] if COMPARE_BOTH else (["v2c"] if USE_V2C else ["raw"])

if DEVICE == "cuda" and not torch.cuda.is_available():
    raise ValueError("Không thấy GPU NVIDIA. Đổi DEVICE = 'cpu'.")
torch.set_num_threads(THREADS)

results = []
for path_text in IMAGE_PATHS:
    path, image = load_image(path_text)
    label = normalize(LABELS[path_text]) if path_text in LABELS else None
    lines = [f"Nhãn: {label}"] if label is not None else []
    inputs = []
    if image.height > 0.6 * image.width:
        lines.append("Lưu ý: ảnh có vẻ không phải một dòng chữ; model chỉ đọc một dòng mỗi ảnh.")
    for name in MODELS:
        runner = get_runner(name, DEVICE)
        for mode in PRE:
            prepared, audit = prepare_image(image, mode)
            tick = time.perf_counter()
            text = runner.predict(prepared)
            seconds = time.perf_counter() - tick
            line = f"{name} / {mode}: {text}   ({seconds:.2f} giây)"
            cer = None
            if label is not None:
                cer = line_cer(label, text)
                line += f"   CER {100 * cer:.1f}%"
            if audit and not audit["bbox_found"]:
                line += "   [V2c không thấy mực, giữ nguyên ảnh]"
            elif v2c_touches_border(audit):
                # Trên 678 ảnh val UIT, khung cắt không lần nào chạm mép ảnh; ảnh chụp có viền tối thì chạm.
                line += "   [cảnh báo: khung cắt V2c chạm mép ảnh, có thể bóng/viền tối bị coi là mực]"
            lines.append(line)
            inputs.append((f"Đầu vào model {name} / {mode}", runner.input_image(prepared)))
            results.append(dict(image=path_text, model=name, pre=mode, prediction=text,
                                cer=cer, seconds=round(seconds, 2)))
    if SHOW_V2C_STEPS:
        v2c_image, v2c_audit = prepare_image(image, "v2c")  # vẽ V2c kể cả khi USE_V2C = False, để so
        show_v2c_result(path.name, image, lines, v2c_image, v2c_audit, inputs)
    else:
        show_result(path.name, image, lines, inputs if SHOW_MODEL_INPUT else ())

for row in results:
    print(row)
